# ESM 3Di Retraining Experiment

This notebook demonstrates how to retrain ESM-2 models for 3Di structure prediction using LoRA (Low-Rank Adaptation).

## Overview
- Load amino acid and 3Di sequence datasets
- Configure LoRA parameters for efficient fine-tuning
- Train the model with per-residue token classification
- Monitor training progress and visualize results
- Run inference on new sequences

## 1. Install and Import Dependencies

In [3]:
# Import core libraries
import os
from typing import List, Tuple
from pathlib import Path

import torch
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import numpy as np

from transformers import AutoTokenizer, AutoModelForTokenClassification
from peft import LoraConfig, get_peft_model, TaskType

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

# Set device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\nUsing device: {device}")

2025-12-01 13:16:26.681683: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-12-01 13:16:38.449943: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
2025-12-01 13:16:38.449943: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


PyTorch version: 2.2.0
CUDA available: False

Using device: cpu


## 2. Utility Functions from esmretrain

Implementing core functions for FASTA reading and data processing.

In [4]:
def read_fasta(path: str) -> List[Tuple[str, str]]:
    """
    Simple FASTA parser.
    Returns list of (header_without_>, sequence_string).
    """
    records = []
    header = None
    seq_chunks = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.startswith(">"):
                if header is not None:
                    records.append((header, "".join(seq_chunks)))
                header = line[1:].strip()
                seq_chunks = []
            else:
                seq_chunks.append(line.strip().upper())
        if header is not None:
            records.append((header, "".join(seq_chunks)))
    return records

print("✓ FASTA reader function defined")

✓ FASTA reader function defined


## 3. Dataset Class

Implementing the custom Dataset class for 3Di sequences with masking support.

In [5]:
class Seq3DiDataset(Dataset):
    """
    Holds (amino_acid_sequence, 3Di_label_sequence) pairs.
    Assumes 1:1 correspondence and equal lengths.

    mask_label_chars: characters in 3Di sequences that indicate "masked" positions
                      (e.g. low pLDDT). Those chars:
                      - are NOT part of label_vocab / model outputs
                      - are ignored in the loss (labels = -100).
    """
    def __init__(
        self,
        aa_fasta: str,
        three_di_fasta: str,
        mask_label_chars: str = ""
    ):
        aa_records = read_fasta(aa_fasta)
        lab_records = read_fasta(three_di_fasta)

        assert len(aa_records) == len(lab_records), "Mismatched number of sequences"

        self.items = []
        all_chars = set()
        self.mask_label_chars = set(mask_label_chars) if mask_label_chars else set()

        for (h_aa, seq_aa), (h_lab, seq_lab) in zip(aa_records, lab_records):
            if len(seq_aa) != len(seq_lab):
                raise ValueError(
                    f"Length mismatch {h_aa}/{h_lab}: {len(seq_aa)} vs {len(seq_lab)}"
                )
            self.items.append((h_aa, seq_aa, seq_lab))
            all_chars.update(seq_lab)

        # Build vocab from all non-masked characters
        label_chars = sorted(ch for ch in all_chars if ch not in self.mask_label_chars)
        self.label_vocab = label_chars
        self.char2idx = {c: i for i, c in enumerate(self.label_vocab)}

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        # (header, aa_seq, 3di_seq)
        return self.items[idx]

print("✓ Seq3DiDataset class defined")

✓ Seq3DiDataset class defined


## 4. Collate Function

Custom collate function for DataLoader that handles tokenization and label alignment.

In [6]:
def make_collate_fn(tokenizer, char2idx, mask_label_chars: str = ""):
    """
    - Tokenizes AA sequences with HF ESM tokenizer.
    - Uses special_tokens_mask to align per-residue 3Di labels
      to non-special tokens.
    - Positions whose 3Di label is in mask_label_chars are set to -100
      (ignored in the loss) and do NOT belong to label_vocab.
    """
    mask_set = set(mask_label_chars) if mask_label_chars else set()

    def collate(batch):
        headers, aa_seqs, label_seqs = zip(*batch)

        enc = tokenizer(
            list(aa_seqs),
            return_tensors="pt",
            padding=True,
            truncation=True,
            add_special_tokens=True,
            return_special_tokens_mask=True,
        )
        input_ids = enc["input_ids"]              # [B, T]
        attention_mask = enc["attention_mask"]    # [B, T]
        special_mask = enc["special_tokens_mask"] # [B, T]

        batch_size, max_len = input_ids.shape
        labels = torch.full(
            (batch_size, max_len),
            -100,  # ignore_index for CE
            dtype=torch.long,
        )

        for i, lab_seq in enumerate(label_seqs):
            k = 0  # index into label sequence
            for j in range(max_len):
                if special_mask[i, j] == 1:
                    # CLS/EOS/PAD etc.
                    labels[i, j] = -100
                else:
                    if k < len(lab_seq):
                        ch = lab_seq[k]
                        if ch in mask_set:
                            # masked (e.g. low pLDDT) -> ignore in loss
                            labels[i, j] = -100
                        else:
                            try:
                                labels[i, j] = char2idx[ch]
                            except KeyError:
                                raise ValueError(
                                    f"Label char '{ch}' not in vocabulary and not in "
                                    f"mask_label_chars; check your data."
                                )
                        k += 1
                    else:
                        labels[i, j] = -100
            if k != len(lab_seq):
                # Safety check: we should have consumed all labels
                raise ValueError(
                    f"Did not consume all labels for sequence {headers[i]}: "
                    f"used {k}, have {len(lab_seq)}"
                )

        batch_out = {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
        }
        return batch_out

    return collate

print("✓ Collate function defined")

✓ Collate function defined


## 5. Configuration & Hyperparameters

Set up all training parameters and paths.

In [7]:
# Training configuration
CONFIG = {
    # Data paths (UPDATE THESE WITH YOUR DATA)
    'aa_fasta': 'data/training_data_aa.fasta',
    'three_di_fasta': 'data/training_data_3di_masked.fasta',
    
    # Model
    'hf_model': 'facebook/esm2_t33_650M_UR50D',  # or esm2_t12_35M_UR50D for faster testing
    
    # Masking
    'mask_label_chars': 'X',
    
    # LoRA configuration
    'lora_r': 8,
    'lora_alpha': 16.0,
    'lora_dropout': 0.05,
    'lora_target_modules': ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'fc1', 'fc2'],
    
    # Training
    'batch_size': 2,
    'epochs': 3,
    'learning_rate': 1e-4,
    'weight_decay': 1e-2,
    'num_workers': 0,
    'log_every': 10,
    
    # Output
    'out_dir': 'checkpoints_notebook',
}

print("Configuration:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")

# Create output directory
os.makedirs(CONFIG['out_dir'], exist_ok=True)
print(f"\n✓ Output directory ready: {CONFIG['out_dir']}")

Configuration:
  aa_fasta: data/training_data_aa.fasta
  three_di_fasta: data/training_data_3di_masked.fasta
  hf_model: facebook/esm2_t33_650M_UR50D
  mask_label_chars: X
  lora_r: 8
  lora_alpha: 16.0
  lora_dropout: 0.05
  lora_target_modules: ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'fc1', 'fc2']
  batch_size: 2
  epochs: 3
  learning_rate: 0.0001
  weight_decay: 0.01
  num_workers: 0
  log_every: 10
  out_dir: checkpoints_notebook

✓ Output directory ready: checkpoints_notebook


## 6. Load Dataset

Load the amino acid and 3Di FASTA files and create the dataset.

In [8]:
# Load dataset
dataset = Seq3DiDataset(
    CONFIG['aa_fasta'],
    CONFIG['three_di_fasta'],
    mask_label_chars=CONFIG['mask_label_chars'],
)

print(f"✓ Loaded {len(dataset)} sequences")
print(f"✓ 3Di vocabulary size: {len(dataset.label_vocab)}")
print(f"✓ 3Di alphabet: {dataset.label_vocab}")

if CONFIG['mask_label_chars']:
    print(f"✓ Masked characters (ignored in loss): {list(set(CONFIG['mask_label_chars']))}")

# Show sample
if len(dataset) > 0:
    sample_header, sample_aa, sample_3di = dataset[0]
    print(f"\nSample sequence:")
    print(f"  Header: {sample_header}")
    print(f"  AA length: {len(sample_aa)}")
    print(f"  3Di length: {len(sample_3di)}")
    print(f"  AA (first 50): {sample_aa[:50]}")
    print(f"  3Di (first 50): {sample_3di[:50]}")

FileNotFoundError: [Errno 2] No such file or directory: 'data/training_data_aa.fasta'

## 7. Initialize Model with LoRA

Load the base ESM model and apply LoRA adapters for efficient fine-tuning.

In [ ]:
# Load tokenizer
print(f"Loading tokenizer: {CONFIG['hf_model']}")
tokenizer = AutoTokenizer.from_pretrained(CONFIG['hf_model'])
print("✓ Tokenizer loaded")

# Load base model
print(f"\nLoading base model: {CONFIG['hf_model']}")
base_model = AutoModelForTokenClassification.from_pretrained(
    CONFIG['hf_model'],
    num_labels=len(dataset.label_vocab),
)
print("✓ Base model loaded")

# Configure LoRA
lora_config = LoraConfig(
    task_type=TaskType.TOKEN_CLS,
    r=CONFIG['lora_r'],
    lora_alpha=CONFIG['lora_alpha'],
    lora_dropout=CONFIG['lora_dropout'],
    target_modules=CONFIG['lora_target_modules'],
)

# Apply LoRA
model = get_peft_model(base_model, lora_config)
print("✓ LoRA adapters applied")

# Freeze base weights, only train LoRA and classifier
def freeze_all_but_lora_and_classifier(model):
    for name, p in model.named_parameters():
        p.requires_grad = False
    
    for name, p in model.named_parameters():
        if "lora_" in name or "classifier" in name:
            p.requires_grad = True

freeze_all_but_lora_and_classifier(model)
model.to(device)

# Print trainable parameters
model.print_trainable_parameters()

print(f"\n✓ Model ready on {device}")

## 8. Create DataLoader

Set up the DataLoader with custom collate function.

In [ ]:
# Create collate function
collate_fn = make_collate_fn(
    tokenizer,
    dataset.char2idx,
    mask_label_chars=CONFIG['mask_label_chars'],
)

# Create DataLoader
loader = DataLoader(
    dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=True,
    num_workers=CONFIG['num_workers'],
    collate_fn=collate_fn,
)

print(f"✓ DataLoader created")
print(f"  Batch size: {CONFIG['batch_size']}")
print(f"  Number of batches: {len(loader)}")
print(f"  Total samples: {len(dataset)}")

## 9. Setup Optimizer

Initialize the AdamW optimizer for training.

In [ ]:
# Setup optimizer
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=CONFIG['learning_rate'],
    weight_decay=CONFIG['weight_decay'],
)

print(f"✓ Optimizer configured")
print(f"  Type: AdamW")
print(f"  Learning rate: {CONFIG['learning_rate']}")
print(f"  Weight decay: {CONFIG['weight_decay']}")
print(f"  Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## 10. Training Loop

Main training loop with loss tracking and checkpoint saving.

In [ ]:
# Training history
history = {
    'epoch_losses': [],
    'step_losses': [],
    'steps': [],
}

# Training loop
model.train()
print(f"Starting training for {CONFIG['epochs']} epochs...\n")

for epoch in range(1, CONFIG['epochs'] + 1):
    print(f"{'='*60}")
    print(f"EPOCH {epoch}/{CONFIG['epochs']}")
    print(f"{'='*60}")
    
    epoch_loss = 0.0
    epoch_tokens = 0
    running_loss = 0.0
    running_tokens = 0
    
    for step, batch in enumerate(loader, start=1):
        # Move batch to device
        batch = {k: v.to(device) for k, v in batch.items()}
        
        # Forward pass
        outputs = model(**batch)
        loss = outputs.loss
        
        # Count valid tokens (not ignored)
        token_count = (batch["labels"] != -100).sum().item()
        
        # Backward pass
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        
        # Accumulate metrics
        loss_value = loss.item()
        running_loss += loss_value * max(token_count, 1)
        running_tokens += max(token_count, 1)
        epoch_loss += loss_value * max(token_count, 1)
        epoch_tokens += max(token_count, 1)
        
        # Log periodically
        if step % CONFIG['log_every'] == 0:
            avg_loss = running_loss / max(running_tokens, 1)
            print(f"  Step {step}/{len(loader)} | Loss/residue: {avg_loss:.4f} | Tokens: {running_tokens}")
            
            # Record for plotting
            history['step_losses'].append(avg_loss)
            history['steps'].append((epoch - 1) * len(loader) + step)
            
            running_loss = 0.0
            running_tokens = 0
    
    # Epoch summary
    avg_epoch_loss = epoch_loss / max(epoch_tokens, 1)
    history['epoch_losses'].append(avg_epoch_loss)
    print(f"\n  Epoch {epoch} Complete | Avg Loss/residue: {avg_epoch_loss:.4f}")
    
    # Save checkpoint
    ckpt_path = os.path.join(CONFIG['out_dir'], f"epoch_{epoch}.pt")
    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "label_vocab": dataset.label_vocab,
            "mask_label_chars": CONFIG['mask_label_chars'],
            "args": CONFIG,
            "epoch": epoch,
            "loss": avg_epoch_loss,
        },
        ckpt_path,
    )
    print(f"  ✓ Checkpoint saved: {ckpt_path}\n")

print(f"{'='*60}")
print("TRAINING COMPLETE!")
print(f"{'='*60}")

## 11. Visualize Training Results

Plot the training loss curves to monitor convergence.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot step-wise loss
if history['step_losses']:
    axes[0].plot(history['steps'], history['step_losses'], alpha=0.6, label='Step Loss')
    axes[0].set_xlabel('Training Step')
    axes[0].set_ylabel('Loss per Residue')
    axes[0].set_title('Training Loss (Step-wise)')
    axes[0].grid(True, alpha=0.3)
    axes[0].legend()

# Plot epoch-wise loss
if history['epoch_losses']:
    epochs = list(range(1, len(history['epoch_losses']) + 1))
    axes[1].plot(epochs, history['epoch_losses'], 'o-', linewidth=2, markersize=8)
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Average Loss per Residue')
    axes[1].set_title('Training Loss (Epoch-wise)')
    axes[1].grid(True, alpha=0.3)
    
    # Annotate final loss
    final_loss = history['epoch_losses'][-1]
    axes[1].annotate(f'Final: {final_loss:.4f}',
                     xy=(len(history['epoch_losses']), final_loss),
                     xytext=(10, 10), textcoords='offset points',
                     bbox=dict(boxstyle='round,pad=0.5', fc='yellow', alpha=0.7),
                     arrowprops=dict(arrowstyle='->', connectionstyle='arc3,rad=0'))

plt.tight_layout()
plt.savefig(os.path.join(CONFIG['out_dir'], 'training_curves.png'), dpi=150)
print(f"✓ Training curves saved to {CONFIG['out_dir']}/training_curves.png")
plt.show()

## 12. Inference Function

Implement prediction function to run inference on new sequences.

In [ ]:
@torch.no_grad()
def predict_3di(model, tokenizer, label_vocab, aa_sequence: str, device: str = 'cpu'):
    """
    Predict 3Di sequence for a given amino acid sequence.
    
    Args:
        model: Trained model
        tokenizer: ESM tokenizer
        label_vocab: List of 3Di characters
        aa_sequence: Amino acid sequence string
        device: Device to run on
        
    Returns:
        Predicted 3Di sequence string
    """
    model.eval()
    idx2char = {i: c for i, c in enumerate(label_vocab)}
    
    # Tokenize
    enc = tokenizer(
        aa_sequence,
        return_tensors="pt",
        add_special_tokens=True,
        return_special_tokens_mask=True,
    )
    enc = {k: v.to(device) for k, v in enc.items()}
    
    # Forward pass
    outputs = model(**enc)
    logits = outputs.logits[0]  # [T, num_labels]
    special_mask = enc["special_tokens_mask"][0]  # [T]
    
    # Get predictions
    pred_indices = logits.argmax(dim=-1)
    pred_chars = []
    
    for j in range(pred_indices.shape[0]):
        if special_mask[j] == 1:
            # Skip special tokens
            continue
        pred_chars.append(idx2char[int(pred_indices[j])])
    
    return "".join(pred_chars)

print("✓ Inference function defined")

## 13. Test Inference on Sample Sequences

Run predictions on sample sequences from the training set.

In [ ]:
# Test on a few samples from the dataset
num_samples = min(3, len(dataset))
print(f"Testing inference on {num_samples} samples:\n")

for i in range(num_samples):
    header, aa_seq, true_3di = dataset[i]
    
    # Predict
    pred_3di = predict_3di(model, tokenizer, dataset.label_vocab, aa_seq, device)
    
    # Calculate accuracy (ignoring masked positions)
    if len(pred_3di) == len(true_3di):
        # Count matches (excluding mask characters)
        matches = sum(
            1 for p, t in zip(pred_3di, true_3di)
            if t not in set(CONFIG['mask_label_chars']) and p == t
        )
        valid_positions = sum(
            1 for t in true_3di
            if t not in set(CONFIG['mask_label_chars'])
        )
        accuracy = matches / valid_positions if valid_positions > 0 else 0.0
    else:
        accuracy = 0.0
        print(f"Warning: Length mismatch for {header}")
    
    print(f"Sample {i+1}: {header}")
    print(f"  AA sequence:  {aa_seq[:60]}...")
    print(f"  True 3Di:     {true_3di[:60]}...")
    print(f"  Predicted:    {pred_3di[:60]}...")
    print(f"  Accuracy:     {accuracy*100:.2f}%")
    print()

## 14. Load Checkpoint and Resume

Example of how to load a saved checkpoint for continued training or inference.

In [ ]:
# Example: Load a checkpoint
def load_checkpoint(checkpoint_path, device='cpu'):
    """Load a model checkpoint."""
    print(f"Loading checkpoint: {checkpoint_path}")
    
    ckpt = torch.load(checkpoint_path, map_location=device)
    
    # Extract metadata
    label_vocab = ckpt['label_vocab']
    mask_chars = ckpt.get('mask_label_chars', '')
    saved_config = ckpt['args']
    epoch = ckpt.get('epoch', 0)
    loss = ckpt.get('loss', 0.0)
    
    print(f"  Checkpoint from epoch {epoch}, loss: {loss:.4f}")
    print(f"  Label vocabulary size: {len(label_vocab)}")
    
    # Rebuild model
    tokenizer = AutoTokenizer.from_pretrained(saved_config['hf_model'])
    base_model = AutoModelForTokenClassification.from_pretrained(
        saved_config['hf_model'],
        num_labels=len(label_vocab),
    )
    
    lora_config = LoraConfig(
        task_type=TaskType.TOKEN_CLS,
        r=saved_config['lora_r'],
        lora_alpha=saved_config['lora_alpha'],
        lora_dropout=saved_config['lora_dropout'],
        target_modules=saved_config['lora_target_modules'],
    )
    
    model = get_peft_model(base_model, lora_config)
    model.load_state_dict(ckpt['model_state_dict'])
    model.to(device)
    model.eval()
    
    print("✓ Checkpoint loaded successfully")
    
    return model, tokenizer, label_vocab, mask_chars

# Example usage (uncomment to use):
# checkpoint_path = os.path.join(CONFIG['out_dir'], 'epoch_3.pt')
# loaded_model, loaded_tokenizer, loaded_vocab, _ = load_checkpoint(checkpoint_path, device)
# print(\"Model ready for inference!\")

## 15. Summary and Next Steps

### What we accomplished:
- ✅ Loaded amino acid and 3Di training data with masking support
- ✅ Initialized ESM-2 model with LoRA adapters for efficient fine-tuning
- ✅ Trained the model for per-residue 3Di prediction
- ✅ Monitored training progress with loss tracking
- ✅ Visualized training curves
- ✅ Implemented inference function for predictions
- ✅ Saved checkpoints for future use

### Next Steps:
1. **Evaluate on validation set**: Split your data and track validation metrics
2. **Hyperparameter tuning**: Experiment with learning rate, LoRA rank, batch size
3. **Longer training**: Increase epochs for better convergence
4. **Model size**: Try different ESM-2 variants (t12_35M, t30_150M, t33_650M)
5. **Data augmentation**: Add more training sequences from AlphaFold DB
6. **Create FoldSeek database**: Use trained model with `fastas2foldseekdb.py`

### Using the trained model:
```python
# Load checkpoint
model, tokenizer, vocab, _ = load_checkpoint('checkpoints_notebook/epoch_3.pt', device)

# Predict on new sequence
new_sequence = "MKTAYIAKQRQISFVKSHFSRQLE"
prediction = predict_3di(model, tokenizer, vocab, new_sequence, device)
print(f"Predicted 3Di: {prediction}")
```